# 👑 StockGen AI - The Ultimate Pipeline (BiRefNet + Smart Upscale)
สมุดงานนี้คือ "เวอร์ชันสมบูรณ์แบบ" ที่รวมเอาความแม่นยำระดับท็อปของ `birefnet-general` มารวมกับความเร็วของ `Smart Pipeline` เพื่อให้ได้งานที่เนียนที่สุด เร็วที่สุด และไม่กิน RAM จนล้น

**Flow การทำงาน:**
1. ลบพื้นหลังและเกลี่ยขอบบน **ภาพต้นฉบับ** ด้วย `birefnet-general` + Alpha Matting
2. ดึงเฉพาะหน้ากาก (Alpha Mask) ออกมา
3. อัปสเกลภาพต้นฉบับเป็น 4K ด้วย RealESRGAN
4. ขยายหน้ากากเป็น 4K ด้วยคณิตศาสตร์ (LANCZOS4) แล้วประกบกลับเข้าไป

In [ ]:
!nvidia-smi
!pip install rembg[gpu] realesrgan basicsr opencv-python-headless requests pillow matplotlib

### โหลดรูปและเตรียมโมเดล

In [ ]:
import cv2
import numpy as np
import time
import requests
from PIL import Image
from io import BytesIO
import matplotlib.pyplot as plt
import sys

try:
    import torchvision.transforms.functional_tensor
except ImportError:
    try:
        import torchvision.transforms.functional as functional_tensor
        sys.modules['torchvision.transforms.functional_tensor'] = functional_tensor
    except ImportError:
        pass

# 1. โหลดภาพต้นฉบับ
image_url = 'https://raw.githubusercontent.com/danielgatis/rembg/master/examples/animal-1.jpg'
response = requests.get(image_url)
img_pil = Image.open(BytesIO(response.content)).convert("RGB")
print("📥 โหลดรูปภาพสำเร็จ! ขนาด:", img_pil.size)

# 2. โหลดโมเดล
print("🔄 กำลังโหลดโมเดลอัปสเกล...")
from basicsr.archs.rrdbnet_arch import RRDBNet
from realesrgan import RealESRGANer
model = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64, num_block=23, num_grow_ch=32, scale=4)
upsampler = RealESRGANer(
    scale=4,
    model_path='https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    model=model,
    half=True,
    gpu_id=0
)

print("🔄 กำลังโหลดโมเดล BiRefNet (1GB)...")
from rembg import remove, new_session
session_biref = new_session('birefnet-general')
print("✅ โหลดโมเดลพร้อมลุย!")

### 🚀 รัน Smart Pipeline (BiRefNet + Upscale 4X)

In [ ]:
print("\n=== 🚀 เริ่มต้น The Ultimate Pipeline ===")
t_total = time.time()

# 1. ลบพื้นหลังบนภาพเล็กด้วย BiRefNet (ปิด Alpha Matting)
t0 = time.time()
small_nobg_pil = remove(
    img_pil, 
    session=session_biref, 
    alpha_matting=False
)
small_alpha_np = np.array(small_nobg_pil)[:, :, 3]
print(f"[1] ลบพื้นหลัง (BiRefNet ล้วน) ใช้เวลา: {time.time() - t0:.2f} วิ")

# 2. อัปสเกล 4x บนภาพต้นฉบับ
t0 = time.time()
img_cv2 = np.array(img_pil)[:, :, ::-1]
upscaled_cv2, _ = upsampler.enhance(img_cv2, outscale=4)
print(f"[2] อัปสเกล 4x (รูปต้นฉบับ) สำเร็จ ใช้เวลา: {time.time() - t0:.2f} วิ")

# 3. ขยายแผ่นหน้ากาก และรวมร่าง (พร้อมเทคนิคลบเงาดำ + ขอบนุ่ม)
t0 = time.time()
out_h, out_w = upscaled_cv2.shape[:2]
upscaled_alpha_np = cv2.resize(small_alpha_np, (out_w, out_h), interpolation=cv2.INTER_LINEAR)

# --- เทคนิคขอบนุ่มไร้เงาดำ (ANTI-HALO & SOFT EDGE) ---
# 3.1 หดขอบหน้ากากเข้ามาเล็กน้อยเพื่อตัดเงาดำ
kernel = np.ones((3, 3), np.uint8)
upscaled_alpha_np = cv2.erode(upscaled_alpha_np, kernel, iterations=1)

# 3.2 ใส่ Gaussian Blur เพื่อให้ขอบฟุ้ง (Feathering) ดูเป็นธรรมชาติ
# ใช้ Kernel (3,3) เพื่อความนุ่มที่พอดี ไม่เบลอจนฟุ้งเกินไป
upscaled_alpha_np = cv2.GaussianBlur(upscaled_alpha_np, (3, 3), 0)

final_rgba_np = np.zeros((out_h, out_w, 4), dtype=np.uint8)
final_rgba_np[:, :, :3] = upscaled_cv2[:, :, ::-1] # RGB
final_rgba_np[:, :, 3] = upscaled_alpha_np
final_nobg_pil = Image.fromarray(final_rgba_np, 'RGBA')
print(f"[3] รวมร่าง 4K + เทคนิคขอบนุ่ม สำเร็จ ใช้เวลา: {time.time() - t0:.2f} วิ")

print(f"\n🎉 เสร็จสิ้น! เวลาสุทธิ: {time.time() - t_total:.2f} วิ")
print(f"ขนาดผลลัพธ์: {final_nobg_pil.size}")

### 🔍 ซูมดูความเนียนระดับ 4K

In [ ]:
def display_with_checkerboard(img, max_width=600):
    from PIL import ImageDraw
    bg = Image.new('RGB', img.size, (255, 255, 255))
    draw = ImageDraw.Draw(bg)
    sq_size = 40
    for y in range(0, img.height, sq_size):
        for x in range(0, img.width, sq_size):
            if (x // sq_size + y // sq_size) % 2 == 0:
                draw.rectangle([x, y, x+sq_size, y+sq_size], fill=(200, 200, 200))
    bg.paste(img, (0, 0), img)
    display(bg.resize((max_width, int(max_width * bg.height / bg.width))))

print("✨ 1. ภาพเต็ม (ย่อแล้ว)")
display_with_checkerboard(final_nobg_pil, max_width=400)

print("\n🔍 2. ซูมดูระดับพิกเซล (Pixel-Peeping บริเวณที่มีขน/ขอบซับซ้อน)")
# ครอป 4K ซูมดูใกล้ๆ (คุณเปลี่ยนพิกัดได้ถ้ารูปต่างออกไป)
crop_box = (800, 500, 1500, 1200) # (left, upper, right, lower)
img_crop = final_nobg_pil.crop(crop_box)
display_with_checkerboard(img_crop, max_width=600)